# PC-QAOA, StatePrep PC-QAOA, PenaltyQAOA, and QAOA+ Comparison

This notebook reads the fixed COP manifest and saved results. It does not generate COPs or run optimization.

Mean plots show pointwise 95% Student-$t$ confidence intervals across COPs:

$$\bar y \pm t_{0.975,n-1}\frac{s}{\sqrt{n}}.$$

Here, $n$ is the number of COPs and $s$ is their sample standard deviation. Optimizer restarts and measurement shots are not independent observations. Paired comparisons use per-COP differences. Intervals are omitted when fewer than two COPs contribute.


Performance and runtime results default to the new uncompiled runs. Original compiled runs are retained for the explicit runtime comparison only. Missing uncompiled results are not replaced by smoke or compiled results.


In [ ]:
from pathlib import Path
from collections import Counter
import json, os, sys

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "code-tuning"]
REPO = next(path for path in candidates if (path / "final_experiment/random_cops/config.json").exists())
os.environ.setdefault("MPLCONFIGDIR", str(REPO / "final_experiment/random_cops/.cache/matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Image
from scipy.stats import t

MANIFEST_PATH = Path(os.environ.get(
    "PCQAOA_FINAL_MANIFEST", REPO / "final_experiment/random_cops/artifacts/manifest.json"))
PC_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PC_RESULTS", REPO / "final_experiment/random_cops/results/uncompiled/pcqaoa"))
PENALTY_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_PENALTY_RESULTS", REPO / "final_experiment/random_cops/results/uncompiled/penaltyqaoa"))
QAOA_PLUS_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_QAOA_PLUS_RESULTS", REPO / "final_experiment/random_cops/results/uncompiled/qaoa_plus"))
FIGURE_DIR = Path(os.environ.get(
    "PCQAOA_FINAL_FIGURES", REPO / "final_experiment/random_cops/results/figures"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
VCG_DATABASE_PATH = Path(os.environ.get(
    "PCQAOA_FINAL_VCG_DATABASE", REPO / "final_experiment/random_cops/vcg_results/selected_vcg_db.pkl"))
if not VCG_DATABASE_PATH.exists():
    VCG_DATABASE_PATH = REPO / "tuning/results/full/selected_vcg_db.pkl"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "final_experiment/random_cops"))
sys.path.insert(0, str(REPO / "tuning"))
from figure_style import FigureWriter, METHOD_COLORS, PALETTE, apply_publication_style
import importlib
from final_experiment.random_cops import resource_analysis as final_resource_analysis
final_resource_analysis = importlib.reload(final_resource_analysis)
estimate_final_resources = final_resource_analysis.estimate_final_resources

METHODS = ["PC-QAOA", "PC-QAOA (StatePrep)", "PenaltyQAOA", "QAOA+"]
COLORS = {**METHOD_COLORS, "PC-QAOA (StatePrep)": PALETTE["violet"],
          "Disjoint": PALETTE["blue_secondary"],
          "Overlapping": PALETTE["green_3"],
          "Teal": PALETTE["teal"],
          "Gold": PALETTE["violet"],
          "Gray": PALETTE["neutral_dark"]}
MARKERS = {"PC-QAOA (StatePrep)": "P", "PC-QAOA": "o", "PenaltyQAOA": "s", "QAOA+": "D"}
SIZE_CMAP = LinearSegmentedColormap.from_list(
    "paper_size", [PALETTE["green_1"], PALETTE["teal"], PALETTE["blue_main"]]
)

apply_publication_style()
writer = FigureWriter(FIGURE_DIR, latex=True, display_inline=True)
print("Manifest:", MANIFEST_PATH)
print("PC-QAOA Results:", PC_DIR)
print("PenaltyQAOA Results:", PENALTY_DIR)
print("QAOA+ Results:", QAOA_PLUS_DIR)
STATEPREP_DIR = Path(os.environ.get("PCQAOA_FINAL_STATEPREP_RESULTS",
    REPO / "final_experiment/random_cops/results/uncompiled/pcqaoa_stateprep"))
print("PC-QAOA (StatePrep) Results:", STATEPREP_DIR)
estimate_stateprep_resources = final_resource_analysis.estimate_stateprep_resources


def method_depth_legend(ax, methods, depths, line_styles, depth_markers):
    """Show method colors separately from depth styles."""
    from matplotlib.lines import Line2D
    present = set(methods)
    handles = [Line2D([], [], color=COLORS[method], linewidth=2.5, label=method)
               for method in METHODS if method in present]
    handles += [Line2D([], [], color=COLORS["Gray"],
                       linestyle=line_styles[int(depth)],
                       marker=depth_markers[int(depth)], linewidth=2.0,
                       label=fr"$p={int(depth)}$")
                for depth in sorted(set(depths))]
    ax.legend(handles=handles, frameon=False, fontsize=9, ncol=2)


## Data Loading and Completion

The manifest defines the 500 COPs shared by all four methods. Result files are joined by `instance_id`. A paired comparison includes a COP only when both methods in that comparison have saved the required result.

In [ ]:
def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else default

def distribution_metrics(distribution, reference):
    distribution = np.asarray(distribution, dtype=float)
    total = distribution.sum()
    if total <= 0:
        return {"ar_feas": np.nan}
    distribution = distribution / total
    feasible = np.asarray(reference["feasible_mask"], dtype=bool)
    values = np.asarray(reference["objective_values"], dtype=float)
    p_feasible = float(distribution[feasible].sum())
    if p_feasible <= 0:
        return {"ar_feas": 0.0}
    feasible_objective = float(np.dot(distribution[feasible], values[feasible]) / p_feasible)
    denominator = float(reference["f_max_feasible"] - reference["f_star"])
    ar_feas = ((float(reference["f_max_feasible"]) - feasible_objective) / denominator
               if abs(denominator) > 1e-12 else np.nan)
    return {"ar_feas": ar_feas}

def load_results(directory, method, references):
    rows, depths = [], []
    directory = Path(directory)
    for path in sorted(directory.glob("final_*/result.json")):
        rows.append(load_json(path))
    for path in sorted(directory.glob("final_*/depths.json")):
        instance_id = path.parent.name
        reference = references[instance_id]
        metadata = {
            "n_x": reference["n_x"], "support": reference["support"],
            "regime": reference["regime"], "m": reference["m"], "o": reference["o"],
            "feasible_fraction": reference["feasible_fraction"], "delta": reference["delta"],
            "manuscript_delta": 5 + 2 * abs(reference["f_min_unconstrained"]),
        }
        for row in load_json(path, []):
            sampled = distribution_metrics(row["counts"], reference)
            exact = distribution_metrics(row["probabilities"], reference)
            derived = {**sampled, **{f"exact_{key}": value for key, value in exact.items()}}
            depths.append({
                "instance_id": instance_id, "method": method, **metadata, **derived,
                **{key: value for key, value in row.items()
                   if key not in ["angles", "history", "restarts", "counts", "probabilities"]},
            })
    return pd.DataFrame(rows), pd.DataFrame(depths)

def completion(directory, expected=None):
    statuses = Counter()
    directory = Path(directory)
    selection = load_json(directory / "selection.json")
    if expected is None:
        expected = len(selection["instance_ids"]) if selection else 500
    for path in directory.glob("final_*/status.json"):
        statuses[load_json(path)["status"]] += 1
    completed = len(list(directory.glob("final_*/result.json")))
    statuses["pending"] = max(0, expected - sum(statuses.values()))
    return {"expected": expected, "completed_results": completed,
            "statuses": dict(statuses)}

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Generate or copy the local manifest before using this notebook.")
manifest = load_json(MANIFEST_PATH)
references = {row["id"]: row for row in manifest["instances"]}
instances = pd.DataFrame(manifest["instances"])
instances["manuscript_delta"] = 5 + 2 * instances["f_min_unconstrained"].abs()
pc, pc_depths = load_results(PC_DIR, "PC-QAOA", references)
stateprep, stateprep_depths = load_results(STATEPREP_DIR, "PC-QAOA (StatePrep)", references)
penalty, penalty_depths = load_results(PENALTY_DIR, "PenaltyQAOA", references)
qaoa_plus, qaoa_plus_depths = load_results(QAOA_PLUS_DIR, "QAOA+", references)
for frame, label in [(pc, "PC-QAOA"), (stateprep, "PC-QAOA (StatePrep)"), (penalty, "PenaltyQAOA"),
                     (qaoa_plus, "QAOA+")]:
    if not frame.empty:
        frame["method_label"] = label
for frame in [pc_depths, stateprep_depths, penalty_depths, qaoa_plus_depths]:
    if not frame.empty:
        frame["method_label"] = frame["method"]
result_frames = [frame for frame in [pc, stateprep, penalty, qaoa_plus] if not frame.empty]
depth_frames = [frame for frame in [pc_depths, stateprep_depths, penalty_depths, qaoa_plus_depths] if not frame.empty]
results = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()
depths = pd.concat(depth_frames, ignore_index=True) if depth_frames else pd.DataFrame()

print("Manifest COPs:", len(instances))
print("PC-QAOA (StatePrep):", json.dumps(completion(STATEPREP_DIR), indent=2))
print("PC-QAOA:", json.dumps(completion(PC_DIR), indent=2))
print("PenaltyQAOA:", json.dumps(completion(PENALTY_DIR), indent=2))
print("QAOA+:", json.dumps(completion(QAOA_PLUS_DIR), indent=2))
display(instances[["id", "vcg_id", "vcg_family", "support", "n_x", "m", "o",
                   "n_structural", "n_penalty", "delta"]].head())

## Dataset Composition

Each COP contains one trained $L_{\mathrm{fid}}$ VCG constraint and one or two exactly structurally enforceable constraints. The objective coefficients are integers from $-5$ through $5$. PenaltyQAOA uses

$$\delta_{\mathrm{range}}=U_Q-L_Q+1,$$

where $L_Q$ and $U_Q$ are the sums of the negative and positive QUBO coefficients, respectively.

PC-QAOA (StatePrep) replaces each trained VCG circuit with PennyLane StatePrep for the exact uniform superposition of that constraint’s local feasible states. It retains the existing exact preparation circuits, structural partition, penalties, slack registers, Grover mixers, and optimization settings. The purple curves identify this variant.

In [ ]:
dataset_plots = [
    (instances["support"].value_counts().sort_index(), "VCG Support Size", "COPs",
     "COP Count vs. VCG Support Size", COLORS["PC-QAOA"], "dataset_vcg_support"),
    (instances["vcg_family"].value_counts().sort_index(), "VCG Constraint Family", "COPs",
     "COP Count vs. VCG Constraint Family", COLORS["PenaltyQAOA"], "dataset_vcg_family"),
    (instances["n_x"].value_counts().sort_index(), r"Problem Size ($n$)", "COPs",
     r"COP Count vs. Problem Size ($n$)", COLORS["Teal"], "dataset_problem_size"),
]
for values, xlabel, ylabel, title, color, name in dataset_plots:
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    values.plot.bar(ax=ax, color=color)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    if name == "dataset_vcg_family":
        ax.tick_params(axis="x", labelrotation=25)
    else:
        ax.tick_params(axis="x", labelrotation=0)
    writer.save(fig, name)

cell = instances.groupby(["m", "o"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(6.8, 4.0))
cell.plot.bar(ax=ax, color=[PALETTE["neutral"], PALETTE["blue_secondary"], PALETTE["violet"]])
ax.set(xlabel="Constraints per COP", ylabel="COPs",
       title="COP Count vs. Constraints per COP")
ax.tick_params(axis="x", labelrotation=0)
ax.legend(title="Support Overlaps", frameon=False)
writer.save(fig, "dataset_constraint_overlap")

exact_counts = Counter(category for values in instances["exact_categories"] for category in values)
partitions = instances.groupby(["n_structural", "n_penalty"]).size().rename("COPs").reset_index()
display(pd.Series(exact_counts, name="Occurrences").sort_index().to_frame())
display(partitions.rename(columns={"n_structural": "Structurally Enforced",
                                   "n_penalty": "Penalized"}))

## Optimization Metrics

For the feasible set $\mathcal{F}$ and the set of optimal feasible strings $\mathcal X^\star$,

$$
P(\mathrm{feas})=\sum_{x\in\mathcal{F}}P(x),\qquad
P(\mathrm{opt})=\sum_{x\in\mathcal X^\star}P(x).
$$

Slack registers are marginalized before these decision-level probabilities are calculated. As defined in the manuscript, the feasible-state objective and feasible approximation ratio are

$$
\bar f_{\mathrm{feas}}=
\frac{\sum_{x\in\mathcal{F}}f(x)P(x)}{P(\mathrm{feas})},\qquad
\mathrm{AR}_{\mathrm{feas}}
=\frac{\bar f_{\mathrm{feas}}-f_{\max}^{\mathcal{F}}}
{f_{\min}^{\mathcal{F}}-f_{\max}^{\mathcal{F}}}.
$$

The notebook assigns $\mathrm{AR}_{\mathrm{feas}}=0$ when $P(\mathrm{feas})=0$. Sampled metrics use 10,000 shots. Exact statevector probabilities are saved as diagnostics.

In [ ]:
def mean_ci(frame, group, value):
    stats = frame.groupby(group)[value].agg(["mean", "std", "count"])
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    stats["ci95"] = t.ppf(0.975, stats["count"] - 1) * stats["sem"]
    stats.loc[stats["count"] < 2, "ci95"] = np.nan
    return stats.reset_index()

def depth_plot(frame, value, ylabel, title, name, reference=None):
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in frame.groupby("method_label", sort=False):
        stats = mean_ci(subset, "depth", value)
        ax.errorbar(stats["depth"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], marker=MARKERS[method], linewidth=2.0,
                    capsize=3, label=method)
    if reference is not None:
        ax.axhline(reference, color=COLORS["Gray"], linestyle="--", linewidth=1.8)
    ax.set(xlabel=r"Depth ($p$)", ylabel=ylabel, title=title)
    if len(frame):
        ax.set_xticks(sorted(frame["depth"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, name)

if depths.empty:
    print("No saved depth results are available.")
else:
    depth_plot(depths, "p_optimal", r"$P(\mathrm{opt})$",
               r"$P(\mathrm{opt})$ vs. Depth ($p$)", "p_optimal_by_depth")
    depth_plot(depths, "p_feasible", r"$P(\mathrm{feas})$",
               r"$P(\mathrm{feas})$ vs. Depth ($p$)", "p_feasible_by_depth")
    depth_plot(depths, "ar_feas", r"$\mathrm{AR}_{\mathrm{feas}}$",
               r"$\mathrm{AR}_{\mathrm{feas}}$ vs. Depth ($p$)", "ar_feas_by_depth")
    display(mean_ci(depths, ["method_label", "depth"], "p_optimal"))

## Metrics by Problem Size and Constraint Structure

The size plots show $p=1$ and $p=5$ for each method. The overlap plot uses $o=0$ for disjoint supports and $o\in\{1,2\}$ for one or two support overlaps. All intervals are pointwise 95% Student-$t$ intervals across COPs.

In [ ]:
target_depths = [depth for depth in [1, 5] if depth in set(depths.get("depth", []))]
if depths.empty or not target_depths:
    print("No depth-one or depth-five results are available.")
else:
    selected = depths[depths["depth"].isin(target_depths)].copy()
    metrics = [
        ("ar_feas", r"$\mathrm{AR}_{\mathrm{feas}}$", "ar_feas_by_size"),
        ("p_feasible", r"$P(\mathrm{feas})$", "p_feasible_by_size"),
        ("p_optimal", r"$P(\mathrm{opt})$", "p_optimal_by_size"),
    ]
    line_styles = {1: "--", 5: "-"}
    depth_markers = {1: "^", 5: "o"}
    for metric, label, name in metrics:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        for (method, depth), subset in selected.groupby(["method_label", "depth"], sort=False):
            stats = mean_ci(subset, "n_x", metric)
            ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                        color=COLORS[method], marker=depth_markers[int(depth)],
                        linestyle=line_styles[int(depth)], linewidth=2.0, capsize=3,
                        label=fr"{method}, $p={int(depth)}$")
        ax.set(xlabel=r"Problem Size ($n$)", ylabel=label,
               title=fr"{label} vs. Problem Size ($n$)")
        ax.set_xticks(sorted(instances["n_x"].unique()))
        method_depth_legend(ax, selected["method_label"].unique(),
                            selected["depth"].unique(), line_styles, depth_markers)
        writer.save(fig, name)

    final_depth = 5 if 5 in target_depths else max(target_depths)
    final = selected[selected["depth"] == final_depth].copy()
    overlap_labels = {0: r"Disjoint ($o=0$)", 1: r"Overlap Level $o=1$",
                      2: r"Overlap Level $o=2$"}
    overlap_styles = {0: "-", 1: "--", 2: ":"}
    overlap_markers = {0: "o", 1: "s", 2: "D"}
    method_names = {"PC-QAOA (StatePrep)": "pcqaoa_stateprep", "PC-QAOA": "pcqaoa", "PenaltyQAOA": "penaltyqaoa", "QAOA+": "qaoa_plus"}
    for method in METHODS:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        subset = final[final["method_label"] == method]
        if subset.empty:
            ax.text(0.5, 0.5, f"No Completed $p={final_depth}$ Results",
                    ha="center", va="center", transform=ax.transAxes)
        for overlap, group in subset.groupby("o"):
            stats = mean_ci(group, "n_x", "p_optimal")
            ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                        color=COLORS[method], linestyle=overlap_styles[int(overlap)],
                        marker=overlap_markers[int(overlap)], linewidth=2.0, capsize=3,
                        label=overlap_labels[int(overlap)])
        ax.set(xlabel=r"Problem Size ($n$)", ylabel=r"$P(\mathrm{opt})$",
               title=fr"$P(\mathrm{{opt}})$ vs. Problem Size ($n$): {method}, $p={final_depth}$")
        ax.set_xticks(sorted(instances["n_x"].unique()))
        if not subset.empty:
            ax.legend(frameon=False, fontsize=9)
        writer.save(fig, f"{method_names[method]}_p_optimal_by_overlap")

    threshold = (depths.assign(reached=depths["p_feasible"] >= 0.75)
                 .groupby(["method_label", "depth"])["reached"]
                 .agg(["mean", "count"]).reset_index())
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in threshold.groupby("method_label", sort=False):
        ax.plot(subset["depth"], subset["mean"], color=COLORS[method],
                marker=MARKERS[method], linewidth=2.0, label=method)
    ax.set(xlabel=r"Depth ($p$)",
           ylabel=r"Fraction with $P(\mathrm{feas})\geq 0.75$",
           title=r"Threshold-Attainment Fraction vs. Depth ($p$)", ylim=(-0.02, 1.02))
    ax.set_xticks(sorted(depths["depth"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, "feasibility_threshold_by_depth")

    display(final.groupby(["method_label", "regime"])
            [["ar_feas", "p_feasible", "p_optimal"]].agg(["mean", "std", "count"]))

## Paired Final-Depth Comparisons

Each panel compares two methods on COPs completed by both methods at the final common depth. The dashed line marks equal $P(\mathrm{opt})$. The table reports the paired mean difference, defined as the first method minus the second method.

In [ ]:
from itertools import combinations

if depths.empty:
    print("No paired depth results are available.")
else:
    final_depth = 5 if (depths["depth"] == 5).any() else int(depths["depth"].max())
    final = depths[depths["depth"] == final_depth]
    wide = final.pivot(index="instance_id", columns="method_label", values="p_optimal")
    available = [method for method in METHODS if method in wide.columns]
    pairs = list(combinations(available, 2))
    if not pairs:
        print("Results from at least two methods are required for paired comparisons.")
    else:
        metadata = instances.set_index("id")[["n_x", "regime", "support", "m", "o"]]
        summary_rows = []
        method_names = {"PC-QAOA (StatePrep)": "pcqaoa_stateprep", "PC-QAOA": "pcqaoa", "PenaltyQAOA": "penaltyqaoa", "QAOA+": "qaoa_plus"}
        for left, right in pairs:
            paired = wide[[left, right]].dropna().join(metadata)
            if paired.empty:
                continue
            difference = paired[left] - paired[right]
            ci = (t.ppf(0.975, len(difference) - 1) * difference.std(ddof=1) /
                  np.sqrt(len(difference))) if len(difference) > 1 else np.nan
            summary_rows.append({"First Method": left, "Second Method": right,
                                 "COPs": len(paired), "Mean Difference": difference.mean(),
                                 "95% CI Half-Width": ci})
            fig, ax = plt.subplots(figsize=(5.2, 4.2))
            points = ax.scatter(paired[right], paired[left], c=paired["n_x"],
                                cmap=SIZE_CMAP, s=44, alpha=0.80)
            fig.colorbar(points, ax=ax, label=r"Problem Size ($n$)")
            bound = max(paired[[left, right]].to_numpy().max(), 1e-3)
            ax.plot([0, bound], [0, bound], "--", color=COLORS["Gray"], linewidth=1.8)
            ax.set(xlabel=fr"{right} $P(\mathrm{{opt}})$",
                   ylabel=fr"{left} $P(\mathrm{{opt}})$",
                   title=fr"{left} $P(\mathrm{{opt}})$ vs. {right} $P(\mathrm{{opt}})$ at $p={final_depth}$")
            writer.save(fig, f"paired_{method_names[left]}_vs_{method_names[right]}")
        display(pd.DataFrame(summary_rows))

## Feasibility Diagnostics

Each figure compares $P(\mathrm{feas})$ with the fraction of feasible decision strings for one method at $p=5$. The dashed diagonal is the feasible probability produced by a uniform distribution over decision strings. Points above it assign more probability to feasible strings than uniform sampling.

In [ ]:
final_depth = 5 if not depths.empty and (depths["depth"] == 5).any() else (
    int(depths["depth"].max()) if not depths.empty else None)
method_final = {
    method: (depths[(depths["method_label"] == method) &
                    (depths["depth"] == final_depth)].copy()
             if final_depth is not None else pd.DataFrame())
    for method in METHODS
}
method_names = {"PC-QAOA (StatePrep)": "pcqaoa_stateprep", "PC-QAOA": "pcqaoa", "PenaltyQAOA": "penaltyqaoa", "QAOA+": "qaoa_plus"}
for method in METHODS:
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    subset = method_final[method]
    if subset.empty:
        ax.text(0.5, 0.5, f"No Completed $p={final_depth}$ Results",
                ha="center", va="center", transform=ax.transAxes)
    else:
        points = ax.scatter(subset["feasible_fraction"], subset["p_feasible"],
                            c=subset["n_x"], cmap=SIZE_CMAP, s=44, alpha=0.80)
        fig.colorbar(points, ax=ax, label=r"Problem Size ($n$)")
        ax.plot([0, 1], [0, 1], "--", color=COLORS["Gray"], linewidth=1.8)
    ax.set(xlabel="Feasible Fraction", ylabel=r"$P(\mathrm{feas})$",
           title=fr"{method} $P(\mathrm{{feas}})$ vs. Feasible Fraction")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    writer.save(fig, f"{method_names[method]}_p_feasible_by_feasible_fraction")

penalty_final = method_final["PenaltyQAOA"]
if not penalty_final.empty:
    display(penalty_final[["instance_id", "n_x", "n_total", "delta", "manuscript_delta",
                           "feasible_fraction", "p_feasible", "p_optimal",
                           "ar_feas"]].describe().T)

## QAOA+ Mixer Construction

QAOA+ starts from the lexicographically first feasible decision string. The local mixer uses controlled single-bit flips and controlled XY swaps. If the resulting feasible-state graph is disconnected, exact two-state rotations connect its components. The audit below is computed from the fixed feasible sets and optimal strings. It does not run QAOA optimization.


In [ ]:
def local_mixer_audit(instance):
    feasible = instance["feasible_bitstrings"]
    feasible_set = set(feasible)
    n_x = int(instance["n_x"])
    adjacency = {state: set() for state in feasible}
    for state in feasible:
        for wire in range(n_x):
            candidate = list(state)
            candidate[wire] = "1" if candidate[wire] == "0" else "0"
            candidate = "".join(candidate)
            if candidate in feasible_set:
                adjacency[state].add(candidate)
        for left in range(n_x):
            for right in range(left + 1, n_x):
                if state[left] == state[right]:
                    continue
                candidate = list(state)
                candidate[left], candidate[right] = candidate[right], candidate[left]
                candidate = "".join(candidate)
                if candidate in feasible_set:
                    adjacency[state].add(candidate)
    remaining = set(feasible)
    components = []
    while remaining:
        start = min(remaining)
        seen, frontier = {start}, [start]
        while frontier:
            current = frontier.pop()
            for neighbor in adjacency[current]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    frontier.append(neighbor)
        components.append(seen)
        remaining.difference_update(seen)
    initial = min(feasible)
    initial_component = next(component for component in components if initial in component)
    optimum_reachable = bool(initial_component.intersection(instance["optimal_bitstrings"]))
    return {"instance_id": instance["id"], "n_x": n_x,
            "local_component_count": len(components),
            "local_connected": len(components) == 1,
            "bridge_count": len(components) - 1,
            "initial_component_fraction": len(initial_component) / len(feasible),
            "local_optimum_reachable": optimum_reachable}

mixer_audit = pd.DataFrame(local_mixer_audit(row) for row in manifest["instances"])
print("Locally Connected COPs:", int(mixer_audit["local_connected"].sum()), "/", len(mixer_audit))
print("COPs Requiring Bridge Rotations:", int((mixer_audit["bridge_count"] > 0).sum()))
print("COPs with a Local Path from the Initial State to an Optimum:",
      int(mixer_audit["local_optimum_reachable"].sum()), "/", len(mixer_audit))
display(mixer_audit.describe().T)

connected = mixer_audit.groupby("n_x")["local_connected"].mean().reset_index()
fig, ax = plt.subplots(figsize=(6.8, 4.0))
ax.plot(connected["n_x"], connected["local_connected"],
        color=COLORS["QAOA+"], marker=MARKERS["QAOA+"], linewidth=2.0)
ax.set(xlabel=r"Problem Size ($n$)", ylabel="Fraction of COPs",
       title=r"Local Mixer Connectivity vs. Problem Size ($n$)", ylim=(-0.02, 1.02))
ax.set_xticks(sorted(instances["n_x"].unique()))
writer.save(fig, "qaoa_plus_local_mixer_connectivity")

bridges = mean_ci(mixer_audit, "n_x", "bridge_count")
fig, ax = plt.subplots(figsize=(6.8, 4.0))
ax.errorbar(bridges["n_x"], bridges["mean"], yerr=bridges["ci95"].fillna(0),
            color=COLORS["QAOA+"], marker=MARKERS["QAOA+"], linewidth=2.0, capsize=3)
ax.set(xlabel=r"Problem Size ($n$)", ylabel="Bridge Rotations",
       title=r"Bridge Rotations vs. Problem Size ($n$)")
ax.set_xticks(sorted(instances["n_x"].unique()))
writer.save(fig, "qaoa_plus_bridge_rotations")

## Circuit Resources and Runtime

The resource estimates use the fixed constraint partition and the saved $L_{\mathrm{fid}}$ VCGs for every COP. They count the circuits executed by all four methods at $p=1$ and $p=5$. 

- NISQ counts use $\{R_X,R_Y,R_Z,H,\mathrm{CNOT},X,Y,Z,T,S\}$.
- FTQC counts synthesize rotations into $\{H,T,\mathrm{CNOT},S\}$.
- Logical register qubits include decision and slack qubits. Total wires also include work qubits introduced by the gate decomposition.
- QAOA+ counts include the multi-controlled decomposition of its local and bridge moves. They describe this implementation and are not a lower bound for every possible feasibility-preserving mixer.
- The counts do not include error correction, hardware connectivity, or SWAP routing.
- Curves show means and pointwise 95% Student-$t$ intervals across COPs.
In the runtime scatter plot, the three methods are placed at fixed offsets around each integer qubit count. The x-axis ticks give the actual number of simulated qubits.


StatePrep resource counts use its decomposed rotation/CNOT circuit. Preparation is counted once initially and twice per Grover mixer per layer. Its resource cache is separate from the existing three-method cache.

In [ ]:
resource_frame = estimate_final_resources(
    MANIFEST_PATH, VCG_DATABASE_PATH,
    cache_dir=REPO / "final_experiment/random_cops/results/resource_estimates",
    depths=(1, 5),
)
stateprep_resources = estimate_stateprep_resources(
    MANIFEST_PATH,
    cache_dir=REPO / "final_experiment/random_cops/results/stateprep_resource_estimates",
    depths=(1, 5),
)
resource_frame = pd.concat([resource_frame, stateprep_resources], ignore_index=True)
print("Resource Rows:", len(resource_frame))
display(resource_frame.groupby(["method", "gate_set", "depth"])[
    ["logical_qubits", "total_wires", "total_gates", "two_qubit_gates", "t_gates"]
].agg(["mean", "std", "count"]))

qubits = (resource_frame[(resource_frame["gate_set"] == "nisq") &
                          (resource_frame["depth"] == 1)]
          .drop_duplicates(["instance_id", "method"]))
for metric, ylabel, title, name in [
    ("logical_qubits", "Qubits", r"Logical Register Qubits vs. Problem Size ($n$)",
     "logical_qubits_by_size"),
    ("total_wires", "Qubits", r"Total Decomposition Wires vs. Problem Size ($n$)",
     "decomposition_wires_by_size"),
]:
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for method, subset in qubits.groupby("method", sort=False):
        stats = mean_ci(subset, "n_x", metric)
        ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], marker=MARKERS[method], linewidth=2.0,
                    capsize=3, label=method)
    ax.set(xlabel=r"Problem Size ($n$)", ylabel=ylabel, title=title)
    ax.set_xticks(sorted(instances["n_x"].unique()))
    ax.legend(frameon=False)
    writer.save(fig, name)

resource_styles = {1: "--", 5: "-"}
resource_markers = {1: "^", 5: "o"}
resource_plots = [
    ("nisq", "total_gates", "Total Gates", "total_gates_nisq"),
    ("nisq", "two_qubit_gates", "CNOT Gates", "cnot_gates_nisq"),
    ("ftqc", "total_gates", "Total Gates", "total_gates_ftqc"),
    ("ftqc", "t_gates", "T Gates", "t_gates_ftqc"),
]
for gate_set, metric, label, filename in resource_plots:
    subset = resource_frame[resource_frame["gate_set"] == gate_set]
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    for (method, depth), group in subset.groupby(["method", "depth"], sort=False):
        stats = mean_ci(group, "n_x", metric)
        ax.errorbar(stats["n_x"], stats["mean"], yerr=stats["ci95"].fillna(0),
                    color=COLORS[method], linestyle=resource_styles[int(depth)],
                    marker=resource_markers[int(depth)], linewidth=2.0, capsize=3,
                    label=fr"{method}, $p={int(depth)}$")
    ax.set(xlabel=r"Problem Size ($n$)", ylabel=label,
           title=fr"{label} vs. Problem Size ($n$) ({gate_set.upper()})")
    ax.set_xticks(sorted(instances["n_x"].unique()))
    ax.set_yscale("log")
    method_depth_legend(ax, subset["method"].unique(),
                        subset["depth"].unique(), resource_styles, resource_markers)
    writer.save(fig, filename)

if not results.empty:
    method_offsets = {method: offset for method, offset in
                      zip(METHODS, np.linspace(-0.30, 0.30, len(METHODS)))}
    qubit_ticks = np.arange(int(results["n_total"].min()),
                            int(results["n_total"].max()) + 1)
    for metric, label, filename in [
        ("optimize_seconds", "Optimization Time (s)", "optimization_runtime"),
    ]:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        for method, subset in results.groupby("method_label", sort=False):
            # Fixed offsets separate methods while preserving integer qubit counts.
            ax.scatter(subset["n_total"] + method_offsets[method],
                       subset[metric], color=COLORS[method],
                       marker=MARKERS[method], s=22, alpha=0.62, label=method)
        ax.set_xticks(qubit_ticks)
        ax.set_xlim(qubit_ticks[0] - 0.6, qubit_ticks[-1] + 0.6)
        if metric == "compile_seconds":
            ax.set_yscale("log")
        ax.set(xlabel="Simulated Qubits", ylabel=label,
               title=f"{label.removesuffix(' (s)')} vs. Simulated Qubits")
        ax.legend(frameon=False)
        writer.save(fig, filename)
    display(results.groupby("method_label")[["optimize_seconds",
                                              "process_peak_rss_mb"]]
            .agg(["mean", "std", "count"]))

## Reproducibility

`run.json` records the method, configuration, environment, source hashes, VCG database hash, and dataset-manifest hash. `selection.json` records the COP IDs assigned to the run or shard. A resume is rejected when these identifiers do not match the saved run.

In [ ]:
for method, directory in [("PC-QAOA", PC_DIR), ("PC-QAOA (StatePrep)", STATEPREP_DIR),
                          ("PenaltyQAOA", PENALTY_DIR),
                          ("QAOA+", QAOA_PLUS_DIR)]:
    print(f"\n{method}")
    run = load_json(directory / "run.json")
    selection = load_json(directory / "selection.json")
    if run is None:
        print("No run metadata found.")
    else:
        print("Run Identifier:", run["identifier"])
        print("Manifest Hash:", run["manifest_hash"])
        print("Profile:", run["config"]["profile"])
        print("Selected COPs:", len(selection["instance_ids"]) if selection else "Unknown")
print("\nFigure Fonts:", writer.font_status)
print("Figure Directory:", FIGURE_DIR)

## Archived Compiled Runs and Backend Comparison

All preceding performance and runtime figures use the new uncompiled runs. This section reads the original compiled runs separately. They do not replace missing uncompiled observations.

The archived figures report compilation time, optimization time excluding compilation, and performance by depth. The backend comparison joins completed runs of the same COP and method. It compares compiled PennyLane/JAX with the NumPy adjoint implementation; machine differences also affect timing. It does not measure the effect of compilation alone. Performance differences can arise from numerical differences during optimization.


In [ ]:
# Keep archived data separate from the main uncompiled results and depths.
compiled_results_parts, compiled_depth_parts = [], []
for key, method in [("pcqaoa", "PC-QAOA"), ("pcqaoa_stateprep", "PC-QAOA (StatePrep)"),
                    ("penaltyqaoa", "PenaltyQAOA"), ("qaoa_plus", "QAOA+")]:
    summary, history = load_results(REPO / "final_experiment/random_cops/results" / key,
                                    method, references)
    if not summary.empty:
        summary["method_label"] = method
        compiled_results_parts.append(summary)
    if not history.empty:
        compiled_depth_parts.append(history)
compiled_results = pd.concat(compiled_results_parts, ignore_index=True) if compiled_results_parts else pd.DataFrame()
compiled_depths = pd.concat(compiled_depth_parts, ignore_index=True) if compiled_depth_parts else pd.DataFrame()

if not compiled_results.empty:
    offsets = dict(zip(METHODS, np.linspace(-0.30, 0.30, len(METHODS))))
    ticks = np.arange(int(compiled_results.n_total.min()), int(compiled_results.n_total.max()) + 1)
    for metric, label, filename in [
        ("compile_seconds", "Compilation Time (s)", "compiled_compilation_runtime"),
        ("optimize_seconds", "Optimization Time (s)", "compiled_optimization_runtime")]:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        for method, group in compiled_results.groupby("method_label", sort=False):
            valid = group[np.isfinite(group[metric]) & (group[metric] > 0)]
            ax.scatter(valid.n_total + offsets[method], valid[metric], color=COLORS[method],
                       marker=MARKERS[method], s=22, alpha=0.62, label=method)
        ax.set_xticks(ticks)
        ax.set_xlim(ticks[0] - 0.6, ticks[-1] + 0.6)
        if metric == "compile_seconds": ax.set_yscale("log")
        ax.set(xlabel="Simulated Qubits", ylabel=label,
               title=f"{label.removesuffix(' (s)')} vs. Simulated Qubits: Compiled Runs")
        ax.legend(frameon=False)
        writer.save(fig, filename)
    display(compiled_results.groupby("method_label")[["compile_seconds", "optimize_seconds"]]
            .agg(["mean", "median", "count"]))

if not compiled_depths.empty:
    for metric, label, filename in [
        ("p_optimal", r"$P(\mathrm{opt})$", "compiled_p_optimal_by_depth"),
        ("p_feasible", r"$P(\mathrm{feas})$", "compiled_p_feasible_by_depth"),
        ("ar_feas", r"$\mathrm{AR}_{\mathrm{feas}}$", "compiled_ar_feas_by_depth")]:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        for method, group in compiled_depths.groupby("method", sort=False):
            stats = mean_ci(group, "depth", metric)
            ax.errorbar(stats.depth, stats["mean"], yerr=stats.ci95.fillna(0),
                        color=COLORS[method], marker=MARKERS[method], linewidth=2.0,
                        capsize=3, label=method)
        ax.set(xlabel=r"Depth ($p$)", ylabel=label,
               title=label + r" vs. Depth ($p$): Compiled Runs")
        ax.set_xticks(sorted(compiled_depths.depth.unique()))
        ax.legend(frameon=False)
        writer.save(fig, filename)

from final_experiment.random_cops.uncompiled_analysis import load_timing_pairs
from matplotlib.lines import Line2D
uncompiled_pairs = load_timing_pairs(REPO / "final_experiment/random_cops")
if uncompiled_pairs.empty:
    print("No completed uncompiled timing pairs yet. Training is never launched by this notebook.")
else:
    offsets = {method: offset for method, offset in
               zip(METHODS, np.linspace(-0.30, 0.30, len(METHODS)))}
    ticks = np.arange(int(uncompiled_pairs.n_total.min()), int(uncompiled_pairs.n_total.max())+1)
    for comparison in [True]:
        fig, ax = plt.subplots(figsize=(6.8, 4.0))
        for method, subset in uncompiled_pairs.groupby("method", sort=False):
            versions = [("uncompiled_seconds", "o", 0.)] if not comparison else [
                ("compiled_seconds", "^", -.06), ("uncompiled_seconds", "o", .06)]
            for column, marker, shift in versions:
                ax.scatter(subset.n_total+offsets[method]+shift, subset[column],
                           color=COLORS[method], marker=marker if comparison else MARKERS[method],
                           s=22, alpha=.62, label=method if not comparison else None)
        ax.set_xticks(ticks); ax.set_xlim(ticks[0]-.6, ticks[-1]+.6)
        ax.set(xlabel="Simulated Qubits", ylabel="Optimization Time (s)",
               title="Optimization Time vs. Simulated Qubits" +
                     (": Compiled and Uncompiled" if comparison else ": Uncompiled"))
        if comparison:
            handles=[Line2D([],[],color=COLORS[m],linewidth=2,label=m)
                     for m in METHODS if m in set(uncompiled_pairs.method)]
            handles += [Line2D([],[],color=COLORS["Gray"],linestyle="none",marker=marker,label=label)
                        for marker,label in [("^","Compiled"),("o","Uncompiled")]]
            ax.legend(handles=handles, frameon=False, fontsize=9, ncol=2)
        else: ax.legend(frameon=False)
        writer.save(fig, "compiled_vs_uncompiled_optimization_runtime" if comparison else "uncompiled_optimization_runtime")
    display(uncompiled_pairs.groupby("method")[["compiled_seconds","uncompiled_seconds",
            "p_optimal_difference","exact_p_optimal_difference"]].agg(["mean","median","max","count"]))
